# Customer Engagement & Product Utilization Analytics for Retention Strategy
**Prepared by: Aniket Chakraborty**

This notebook reproduces data validation, EDA, segmentation, KPI computation and churn-model evaluation used by the Streamlit application.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from analytics import load_data, add_features, calculate_kpis, grouped_metrics
from modeling import train_and_evaluate, feature_importance

DATA_PATH = Path('data/European_Bank.csv')
df = load_data(DATA_PATH)
analytics_df = add_features(df)
df.head()

## 1. Data Validation

In [ ]:
validation = pd.Series({
    'Rows': len(df),
    'Columns': len(df.columns),
    'Missing values': int(df.isna().sum().sum()),
    'Duplicate rows': int(df.duplicated().sum()),
    'Duplicate customer IDs': int(df.CustomerId.duplicated().sum()),
})
validation

## 2. KPI Summary

In [ ]:
pd.Series(calculate_kpis(analytics_df))

## 3. Churn by Geography

In [ ]:
geo = grouped_metrics(analytics_df, 'Geography')
display(geo)
geo.set_index('Geography')['ChurnRate'].plot(kind='bar', title='Churn Rate by Geography')
plt.ylabel('Churn Rate')
plt.show()

## 4. Engagement Classification

In [ ]:
engagement = grouped_metrics(analytics_df, 'EngagementSegment')
display(engagement.sort_values('ChurnRate', ascending=False))

## 5. Product Utilization

In [ ]:
products = grouped_metrics(analytics_df, 'NumOfProducts')
display(products)
products.set_index('NumOfProducts')['ChurnRate'].plot(kind='bar', title='Churn Rate by Product Count')
plt.ylabel('Churn Rate')
plt.show()

## 6. High-Balance Disengagement

In [ ]:
threshold = analytics_df.Balance.quantile(.75)
premium = analytics_df[analytics_df.Balance >= threshold]
pd.Series({
    'Premium threshold': threshold,
    'Premium customers': len(premium),
    'Inactive premium share': 1 - premium.IsActiveMember.mean(),
    'Inactive premium churn': premium.loc[premium.IsActiveMember.eq(0), 'Exited'].mean(),
})

## 7. Fairness-Aware Churn Model

In [ ]:
model, metrics = train_and_evaluate(df)
pd.Series(metrics)

In [ ]:
feature_importance(model).head(15)

## 8. Recommended Retention Actions

1. Prioritize inactive, high-balance customers for proactive service outreach.
2. Move suitable one-product customers toward a relevant two-product relationship.
3. Investigate the high churn of three- and four-product customers before expanding bundles.
4. Build geography-specific retention pilots, with Germany as the first priority.
5. Use credit-card ownership only as a supporting signal because it has weak standalone separation.
6. Monitor gender outcomes for fairness; do not target interventions solely by gender.